# Backlink Quality Audit with DataForSEO, Embeddings and Claude

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lazarinastoy/mlforseo-code/blob/main/backlink-quality-audit/backlink_quality_audit.ipynb)

By **Lazarina Stoy, [MLforSEO](https://www.mlforseo.com/)**. Free, MIT licence.

Most backlink audits end in one of two places: a scary spam score, or a spreadsheet of a few hundred domains nobody has time to read. This notebook gets you to a short, explainable list instead. It:

1. **Collects your backlink profile** through the DataForSEO Backlinks API: the summary, every referring domain, one representative link per domain, your top anchors and your broken backlinks.
2. **Adds transparent rule flags** you can explain to anyone: spam score band, domain rank, follow or nofollow, spam-prone TLD, where on the page the link sits, anchor type and link age.
3. **Scores topical relevance** with a small embedding model that runs inside the notebook (no API key).
4. **Asks Claude about the ambiguous middle only.** The clear cases are decided by rules; only the rows the rules can't settle go to `claude-haiku-4-5`, in batches, with a strict JSON output.
5. **Sorts every referring domain into four buckets**: Valuable, Neutral, Likely spam and Investigate.
6. **Builds a link reclamation list**: backlinks pointing at pages that return errors, each with a suggested redirect target matched from your sitemap.
7. **Exports** CSVs, one Sheets-ready Excel workbook and a cost log.

**What you need:** a DataForSEO account with Backlinks API access (login + API password from [app.dataforseo.com/api-access](https://app.dataforseo.com/api-access)) and an Anthropic API key ([console.anthropic.com](https://console.anthropic.com/)). You don't need either to try it: with `USE_SAMPLE = True` it replays a real audit of mlforseo.com saved in `sample_data/` (7 November 2026), Claude's answers included.

**What it costs (November 2026 prices):** DataForSEO charges $0.024 per Backlinks API request plus $0.000036 per row returned, so the five calls here cost at most about $0.20 for a site with up to 1,000 referring domains (mlforseo.com: $0.15). Claude Haiku 4.5 is $1 per million input tokens and $5 per million output tokens; reviewing 64 ambiguous links cost $0.024 on mlforseo.com, so 300 would be roughly $0.11.

**A word on spam links before you start.** Google says that in most cases it can work out which links to trust on its own, and that you should only use the disavow tool if you have a considerable number of spammy, artificial or low-quality links *and* they have caused, or are likely to cause, a manual action ([Google Search Central](https://support.google.com/webmasters/answer/2648487)). So the "Likely spam" bucket here is a list to understand and ignore, not a disavow file. The useful parts of a backlink audit are usually the other three buckets and the reclamation list.

## 1. Setup

In Colab this installs the libraries it needs and downloads the helper file and sample data from GitHub. Locally, run it from inside the `backlink-quality-audit/` folder after `pip install -r requirements.txt`.

In [1]:
import json, os, sys, subprocess
from pathlib import Path

REPO_RAW = "https://raw.githubusercontent.com/lazarinastoy/mlforseo-code/main/backlink-quality-audit"
FILES = ["backlink_audit.py", "sample_data/summary.json", "sample_data/referring_domains.json",
         "sample_data/backlinks_one_per_domain.json", "sample_data/anchors.json",
         "sample_data/broken_backlinks.json", "sample_data/claude_reviews.json",
         "sample_data/sitemap_urls.csv", "sample_data/url_status_check.csv"]

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic", "rapidfuzz", "openpyxl",
                    "sentence-transformers"], check=True)
for f in FILES:  # only downloads what isn't already next to the notebook
    if not Path(f).exists():
        Path(f).parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["wget", "-q", "-O", f, f"{REPO_RAW}/{f}"], check=True)

import warnings
warnings.filterwarnings("ignore", message=".*OpenSSL.*")  # harmless urllib3 notice on some Macs
import pandas as pd
import backlink_audit as ba
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)
print("Ready.")

Ready.


## 2. Settings

Leave `USE_SAMPLE = True` for a free dry run on the saved mlforseo.com data. Set it to `False` to audit your own site.

* `TARGET`: your domain, without `https://` or `www.`. `INCLUDE_SUBDOMAINS` also counts links to `blog.yoursite.com` and the like.
* `SITE_TOPIC`: one or two lines about what your site covers. The relevance score compares every link with this sentence, so make it specific.
* `BRAND_TERMS`: words that mean "this anchor names you" (brand, product, founder). `KEYWORD_TERMS`: topic words that make an anchor a keyword anchor.
* `OWNED_DOMAINS`: your other sites and newsletters. Links from them are kept apart, so they don't inflate the Valuable bucket.
* `REVIEW_PATTERNS`: words that send a link to review even if it looks great on paper. Course and template sellers should keep `download`, `sale page`, `group buy` and `course`: sites that pirate or resell paid products look perfectly on-topic to every metric.
* Row limits: up to 1,000 rows per call (DataForSEO's maximum). You're billed per row returned, not per row requested.
* `MAX_SPEND_USD` / `MAX_LLM_SPEND_USD`: hard caps. The clients refuse a call that could take the run over its cap.
* `MAX_LLM_ROWS` and `BATCH_SIZE`: how many ambiguous rows Claude reviews, and how many per call.

In [2]:
USE_SAMPLE = True            # False = call the real APIs with your credentials

TARGET = "mlforseo.com"      # domain without https:// or www.
INCLUDE_SUBDOMAINS = True
SITE_TOPIC = ("machine learning, AI and automation for SEO and marketing: AI search, entity SEO, "
              "keyword research, Python and no-code tutorials, courses")
BRAND_TERMS = ["mlforseo", "ml for seo", "ml4seo", "lazarina", "stoy"]
KEYWORD_TERMS = ["seo", "machine learning", "ml", "ai", "entity", "entities", "keyword", "keywords", "python",
                 "automation", "nlp", "semantic", "search", "marketing", "embeddings", "llm", "course",
                 "glossary", "google sheets", "clustering", "classification", "intent"]
OWNED_DOMAINS = ["lazarinastoy.com", "mlforseo-newsletter.kit.com", "womeninmarketing.bg",
                 "datastudioformarketers.com"]
REVIEW_PATTERNS = ["download", "sale ?page", "group ?buy", "leak", "torrent", "course"]

# DataForSEO Backlinks API
RD_LIMIT = 1000              # referring domains
BL_LIMIT = 1000              # one backlink per referring domain
ANCHOR_LIMIT = 100           # top anchors
BROKEN_LIMIT = 200           # broken backlinks
MAX_SPEND_USD = 0.30         # hard stop for the DataForSEO part of the run

# Claude review of the ambiguous middle
USE_CLAUDE = True
CLAUDE_MODEL = "claude-haiku-4-5"
BATCH_SIZE = 25              # rows per Claude call
MAX_LLM_ROWS = 300           # strongest ambiguous domains first
MAX_LLM_SPEND_USD = 0.25     # hard stop for the Claude part of the run

# Relevance and reclamation
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # small, fast, English. Multilingual: "paraphrase-multilingual-MiniLM-L12-v2"
SITEMAP_URL = "https://www.mlforseo.com/sitemap_index.xml"
CHECK_STATUS_NOW = True      # re-check each dead URL today (free, a few seconds)
RULES = dict(ba.DEFAULT_RULES)  # thresholds, e.g. RULES["rank_valuable"] = 200
RULES

{'spam_score_spam': 60,
 'spam_score_tld': 30,
 'spam_score_clean': 15,
 'rank_valuable': 150,
 'relevance_valuable': 0.3,
 'relevance_floor': 0.1}

## 3. Estimate the cost first

This is a worst-case figure: it assumes every call returns its full row limit. Smaller sites return fewer rows and pay less. The real cost is logged from DataForSEO's own `cost` field after each call.

In [3]:
plan = ba.collection_plan(RD_LIMIT, BL_LIMIT, ANCHOR_LIMIT, BROKEN_LIMIT)
est = ba.estimate_cost(plan)
display(est)
if not USE_SAMPLE and est["usd_max"].iloc[-1] > MAX_SPEND_USD:
    print(f"Heads up: the worst case is above MAX_SPEND_USD (${MAX_SPEND_USD}). "
          "The run stops when the cap is reached. Raise the cap or lower the limits.")

,call,row_limit,usd_max
0,summary,1,0.02404
1,referring_domains,1000,0.06000
2,backlinks,1000,0.06000
3,anchors,100,0.02760
4,broken_backlinks,200,0.03120
5,TOTAL,2301,0.20284


## 4. Connect and collect

In Colab, add two secrets with the key icon in the left sidebar: `DATAFORSEO_LOGIN` and `DATAFORSEO_PASSWORD` (the **API** password, not your dashboard password), and give this notebook access. If they aren't there, you'll get a hidden prompt. Your credentials are never written to the notebook.

The five calls (all `POST https://api.dataforseo.com/v3/backlinks/.../live`, one task each):

| Name | Endpoint | Key parameters |
|---|---|---|
| summary | `summary/live` | `target`, `include_subdomains` |
| referring_domains | `referring_domains/live` | `limit`, `order_by: ["rank,desc"]` |
| backlinks | `backlinks/live` | `mode: "one_per_domain"`, `limit`, `order_by: ["rank,desc"]` |
| anchors | `anchors/live` | `limit`, `order_by: ["backlinks,desc"]` |
| broken_backlinks | `backlinks/live` | `mode: "as_is"`, `filters: ["is_broken", "=", true]` |

All of them count **live** backlinks only (DataForSEO's default `backlinks_status_type`). Ranks are on DataForSEO's 0 to 1,000 scale. Responses are cached in `output/raw/`, so re-running this cell never pays twice for the same request.

In [4]:
if USE_SAMPLE:
    client = ba.SampleClient("sample_data")
    sample_target = ba.first_result(json.loads(Path("sample_data/summary.json").read_text())).get("target")
    if TARGET != sample_target:
        print(f"Sample mode replays {sample_target}; TARGET is ignored until USE_SAMPLE = False.")
    print("Sample mode: replaying saved API responses (mlforseo.com, 7 November 2026)")
else:
    client = ba.DataForSEOClient(*ba.get_credentials(), max_spend_usd=MAX_SPEND_USD)
    print("Connected. Spend cap: $", MAX_SPEND_USD)

raw = ba.collect_profile(client, TARGET, INCLUDE_SUBDOMAINS, RD_LIMIT, BL_LIMIT, ANCHOR_LIMIT, BROKEN_LIMIT)
cost_log = pd.DataFrame(client.cost_log)
print(f"\nDataForSEO spent this run: ${client.spent:.4f}"
      + ("  (sample mode: replaying saved data costs nothing; cost_usd shows what it cost when collected)"
         if USE_SAMPLE else ""))
cost_log

Sample mode: replaying saved API responses (mlforseo.com, 7 November 2026)
  summary            2611 backlinks, 288 referring main domains           cost $0.0240
  referring_domains  total 288, returned 288                              cost $0.0344
  backlinks          total 295, returned 288                              cost $0.0344
  anchors            total 185, returned 100                              cost $0.0276
  broken_backlinks   total 33, returned 33                                cost $0.0252

DataForSEO spent this run: $0.0000  (sample mode: replaying saved data costs nothing; cost_usd shows what it cost when collected)


,endpoint,label,cost_usd,spent_so_far_usd,source
0,summary/live,summary,0.024036,0.0,sample replay (cost when collected)
1,referring_domains/live,referring_domains,0.034368,0.0,sample replay (cost when collected)
2,backlinks/live,backlinks,0.034368,0.0,sample replay (cost when collected)
3,anchors/live,anchors,0.027600,0.0,sample replay (cost when collected)
4,backlinks/live,broken_backlinks,0.025188,0.0,sample replay (cost when collected)


## 5. The profile at a glance

`summary/live` gives the headline numbers and breakdowns across **every** live backlink (not just one per domain). Two things to know when you read it:

* `referring_domains` counts subdomains separately; `referring_main_domains` counts `blog.example.com` and `example.com` as one site. The rest of this notebook works with main domains.
* `backlinks_spam_score` is DataForSEO's spam score (0 to 100) across all your backlinks. A high number usually means a few spam networks with hundreds of links each, not that most of your links are bad. The next cells show which.

In [5]:
display(ba.summary_table(raw["summary"]))
for key in ["referring_links_tld", "referring_links_semantic_locations", "referring_links_attributes"]:
    display(ba.summary_breakdown(raw["summary"], key, top=10))

,metric,value
0,target,mlforseo.com
1,rank,219
2,backlinks,2611
3,referring_domains,611
4,referring_main_domains,288
5,referring_domains_nofollow,498
6,referring_ips,219
7,referring_subnets,159
8,referring_pages,2143
9,backlinks_spam_score,34


,tld,links,share
0,com,906,0.433
1,xyz,743,0.355
2,website,97,0.046
3,space,95,0.045
4,online,93,0.044
5,site,89,0.043
6,co,39,0.019
7,directory,19,0.009
8,org,7,0.003
9,bg,6,0.003


,locations,links,share
0,,1068,0.498
1,main,434,0.203
2,header,243,0.113
3,figure,199,0.093
4,article,95,0.044
5,section,51,0.024
6,footer,40,0.019
7,details,13,0.006


,attributes,links,share
0,nofollow,1173,0.543
1,noopener,553,0.256
2,noreferrer,432,0.200
3,external,1,0.000


## 6. One row per referring domain, with rule features

`backlinks/live` in `one_per_domain` mode returns the strongest link from each referring domain, with `group_count` telling you how many links that domain sends in total. We join the domain-level metrics from `referring_domains/live` and add the features:

| Feature | How it's worked out |
|---|---|
| `spam_band` | DataForSEO `backlink_spam_score`: low (0-14), moderate (15-29), elevated (30-59), high (60-100) |
| `rank_band` | `domain_from_rank` on the 0-1,000 scale |
| `spam_prone_tld` | TLD is in a list of cheap TLDs spam networks favour (`.xyz`, `.top`, `.icu`...). A signal, not a verdict |
| `location_group` | `semantic_location`: content (`main`, `article`, `section`...), boilerplate (`header`, `footer`, `nav`, `aside`) or unknown |
| `anchor_type` | branded, keyword, generic ("click here"), naked URL, image or empty, other |
| `link_age_days` | days since DataForSEO first saw the link |
| `owned` / `review_hint` | from `OWNED_DOMAINS` and `REVIEW_PATTERNS` |

In [6]:
bl = ba.backlinks_df(raw["backlinks"])
rd = ba.referring_domains_df(raw["referring_domains"])
df = ba.add_features(bl, rd, BRAND_TERMS, KEYWORD_TERMS, OWNED_DOMAINS, REVIEW_PATTERNS,
                     today=None if not USE_SAMPLE else "2026-11-07")
print(f"{len(df)} referring domains (one representative link each), {df['group_count'].sum()} links in total")
df[["domain_from", "anchor", "anchor_type", "dofollow", "semantic_location", "domain_from_rank",
    "backlink_spam_score", "spam_band", "tld", "group_count"]].head(10)

288 referring domains (one representative link each), 2596 links in total


,domain_from,anchor,anchor_type,dofollow,semantic_location,domain_from_rank,backlink_spam_score,spam_band,tld,group_count
0,aiseocourse.study,https://www.mlforseo.com/,naked_url,True,section,155,0,low (0-14),study,4
1,loadcourses.com,Sale page,other,True,main,0,0,low (0-14),com,1
2,growthmentor.com,https://www.mlforseo.com/,naked_url,True,section,311,25,moderate (15-29),com,3
3,ipullrank.com,this course.,generic,True,NaN,320,0,low (0-14),com,22
4,womeninmarketing.bg,Link,generic,True,NaN,140,0,low (0-14),bg,6
5,imarketing.courses,Link,generic,True,main,286,5,low (0-14),courses,4
6,moz.com,Intro to Machine Learning,keyword,True,article,540,0,low (0-14),com,2
7,mmocourse.top,Lazarina Stoy – Semantic Keyword Research + AI Search & ...,branded,True,section,359,15,moderate (15-29),top,2
8,aioptimization.services,ML for SEO Academy,branded,True,article,101,5,low (0-14),services,2
9,mmocourse.hk,Lazarina Stoy – Introduction to Machine Learning Course,branded,True,main,381,0,low (0-14),hk,2


### Distributions

One row per referring domain, so a spam network with 60 links still counts once here.

In [7]:
for col in ["spam_band", "rank_band", "anchor_type", "location_group", "dofollow"]:
    display(ba.share_table(df[col], col))
display(ba.share_table(df["tld"], "tld").head(12))

,spam_band,count,share
0,high (60-100),198,0.688
1,low (0-14),60,0.208
2,elevated (30-59),20,0.069
3,moderate (15-29),10,0.035


,rank_band,count,share
0,very low (<50),163,0.566
1,low (50-149),51,0.177
2,medium (150-299),47,0.163
3,high (300+),27,0.094


,anchor_type,count,share
0,spam_phrase,186,0.646
1,naked_url,38,0.132
2,keyword,22,0.076
3,branded,18,0.062
4,other,12,0.042
5,image_or_empty,7,0.024
6,generic,5,0.017


,location_group,count,share
0,content,196,0.681
1,unknown,92,0.319


,dofollow,count,share
0,False,182,0.632
1,True,106,0.368


,tld,count,share
0,com,68,0.236
1,xyz,63,0.219
2,website,27,0.094
3,space,26,0.090
4,online,25,0.087
5,site,24,0.083
6,org,6,0.021
7,co,5,0.017
8,info,4,0.014
9,net,3,0.010


## 7. Topical relevance (a proxy)

Each link is described as `page title | anchor | domain` and compared with `SITE_TOPIC` using `all-MiniLM-L6-v2`, a small sentence-transformers model that runs on a laptop CPU in seconds. The score is the cosine similarity between the two embeddings, clipped to 0-1.

Treat it as a rough proxy, not a judgement. It only sees metadata (never the linking page's text), the model is English-first (titles in Japanese or Portuguese score lower than they should), and spam pages *about SEO* score as "relevant" to an SEO site because they are, topically. That last point is why relevance never decides a link on its own here.

In [8]:
df["relevance"] = ba.embed_relevance(df, SITE_TOPIC, EMBEDDING_MODEL)
bands = pd.cut(df["relevance"], [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 1.0], include_lowest=True)
display(bands.value_counts().sort_index().rename("referring_domains").to_frame())
cols = ["domain_from", "page_from_title", "anchor", "relevance", "backlink_spam_score"]
print("Most relevant:")
display(df.sort_values("relevance", ascending=False)[cols].head(8))
print("Least relevant:")
df.sort_values("relevance")[cols].head(8)

,referring_domains
relevance,
"(-0.001, 0.1]",3
"(0.1, 0.2]",21
"(0.2, 0.3]",133
"(0.3, 0.4]",85
"(0.4, 0.5]",26
"(0.5, 0.6]",16
"(0.6, 1.0]",4


Most relevant:


,domain_from,page_from_title,anchor,relevance,backlink_spam_score
0,aiseocourse.study,AI Search Optimization & Agentic SEO: AI SEO Course Record,https://www.mlforseo.com/,0.723,0
94,completeaitraining.com,11 Recommended AI Courses for SEO Specialists in 2026,AI Search and Agentic SEO Course by Beatrice Gamba,0.712,5
79,aiseocourse.review,"AI SEO Course Cost 2026: Free, One-Time & Monthly Pricing",AI Search Optimization & Agentic SEO,0.674,0
8,aioptimization.services,Semantic Keyword Research: A Practical Guide for Modern SEO,ML for SEO Academy,0.649,5
5,imarketing.courses,Lazarina Stoy – Semantic Keyword Research + AI Search & ...,Link,0.599,5
188,raven-seo.com,Semantic Keyword Research: AI Visibility in 2026,ML for SEO's semantic ML-enabled keyword research material,0.594,0
27,personwise.ai,SEO Metrics Measurement and Interpretation — Free intera...,Your rank tracker is lying to you about AI search - MLfo...,0.590,20
57,aiseocourse.directory,AI SEO course sources | AiSeoCourse Directory,https://academy.mlforseo.com/product/course-bundle-beatr...,0.588,45


Least relevant:


,domain_from,page_from_title,anchor,relevance,backlink_spam_score
253,website.usersupports.com,エンティティSEOを実際にやった — Wikidata QID取得からsameAs実装まで、当サイトのリアル全記...,"MLforSEO ""CrOSS-Platform Entity Consistency""",0.056,15
54,readerfunded.com,"LAZARINA STOY. (lazarinastoy.com), a reader-funded educa...",mlforseo.com,0.079,30
190,brightonseo.com,Lazarina Stoy,NaN,0.087,0
35,newsletter.chuletaseo.com,☕ #ChuletaSEO 123 - Por fin terminó el calvario,asociar palabras clave a categorías,0.102,30
158,way2check.cv,Where to buy 🚀 aged domains and backlinks 🔥 from all-age...,mlforseo.com,0.129,60
22,groas.com,Knowledge Graph Management in AEO: Does It Fix Your Clie...,entity disambiguation,0.131,0
270,tunca.org,Where to buy 🚀 aged domains and backlinks 🔥 | 1025-4941,mlforseo.com,0.135,50
272,globalecommerce.org,Where to buy 🚀 aged domains and backlinks 🔥 from globale...,mlforseo.com,0.137,60


## 8. Rules first: the clear cases

The rules decide what they can and leave the rest as `ambiguous`. In order:

1. `own_site`: the domain is in `OWNED_DOMAINS`.
2. `likely_spam`: spam score at or above 60; or a spam-prone TLD with a spam score of 30+; or a spam score of 30+ with relevance under 0.10.
3. `valuable`: spam score under 15, referring domain rank of 150+, relevance of 0.30+ and no `review_hint`.
4. Everything else: `ambiguous`, sent to Claude.

All thresholds live in `RULES`, so you can tune them for your niche and re-run.

In [9]:
df = ba.apply_rules(df, RULES)
display(ba.share_table(df["rule_bucket"], "rule_bucket"))
display(pd.crosstab(df["rule_bucket"], df["spam_band"]))
df.groupby("rule_bucket")[["relevance", "domain_from_rank", "backlink_spam_score"]].mean().round(2)

,rule_bucket,count,share
0,likely_spam,202,0.701
1,ambiguous,64,0.222
2,valuable,19,0.066
3,own_site,3,0.010


spam_band,elevated (30-59),high (60-100),low (0-14),moderate (15-29)
rule_bucket,,,,
ambiguous,16,0,38,10
likely_spam,4,198,0,0
own_site,0,0,3,0
valuable,0,0,19,0


,relevance,domain_from_rank,backlink_spam_score
rule_bucket,,,
ambiguous,0.36,204.39,14.84
likely_spam,0.28,34.19,65.22
own_site,0.41,311.00,0.00
valuable,0.45,340.42,0.26


## 9. Claude reviews the ambiguous middle

Only the `ambiguous` rows go to Claude (strongest domains first, up to `MAX_LLM_ROWS`), `BATCH_SIZE` per call. Each row is sent as one line of JSON with the link's metadata. The embedding relevance score is deliberately **not** sent: in testing, Claude leaned on that number (and marked good German, Spanish and Japanese articles as low value because the English-first model scored them low), so it now reads the title and anchor itself and gives an independent second opinion. The request uses **structured outputs** (`output_config.format` with a JSON schema), so every answer parses: a `verdict` from `keep_valuable`, `keep_neutral`, `ignore_spam`, `investigate`, a `reason` of 20 words or fewer and a `confidence`.

The prompt tells the model it only sees metadata, not the page, and must not invent facts; when the metadata isn't enough it should say `investigate`. Add `ANTHROPIC_API_KEY` as a Colab secret (or environment variable) before running with `USE_SAMPLE = False`.

This is what one row looks like on its way to Claude:

In [10]:
amb = df[df["rule_bucket"] == "ambiguous"].sort_values("domain_from_rank", ascending=False)
print(ba.estimate_llm_cost(min(len(amb), MAX_LLM_ROWS), BATCH_SIZE, CLAUDE_MODEL))
if len(amb):
    print(ba._row_for_llm(int(amb.index[0]), amb.iloc[0]))

{'rows': 64, 'calls': 3, 'input_tokens_est': 11360, 'output_tokens_est': 2880, 'usd_est': 0.0258}
{'id': 45, 'domain': 'about.me', 'title': 'Claire Broadley - United Kingdom, Red Robot Media Limited, Chartered Institute of Marketing | about.me', 'anchor': 'MLforSEO', 'dofollow': False, 'location': 'section', 'domain_rank': 625, 'spam_score': 0, 'tld': 'me', 'first_seen': '2025-11-06', 'links_from_domain': 2, 'review_hint': None}


In [11]:
print(ba.SYSTEM_PROMPT.format(target=TARGET, site_topic=SITE_TOPIC))

You review backlinks for an SEO audit of the website mlforseo.com.
The site is about: machine learning, AI and automation for SEO and marketing: AI search, entity SEO, keyword research, Python and no-code tutorials, courses

You only see metadata about each link (referring domain, page title, anchor text,
follow status, where on the page the link sits, DataForSEO rank and spam score,
TLD, first-seen date). You do NOT see the linking page itself. Do not invent facts about a site you cannot see; if the
metadata is not enough to decide, say so and choose "investigate".

For each link choose one verdict:
- keep_valuable: a real, relevant site or page that a person might follow to mlforseo.com
  (industry blogs, publications, courses, tools, communities, directories that real people use).
- keep_neutral: real but low value (generic directories, scrapers that copy titles,
  profile pages, aggregators, unrelated but harmless sites). Nothing to do.
- ignore_spam: clear signs of automated spam 

In [12]:
if USE_CLAUDE:
    if USE_SAMPLE:
        reviewer = ba.SampleReviewer("sample_data/claude_reviews.json")
    else:
        reviewer = ba.ClaudeReviewer(ba.get_anthropic_key(), CLAUDE_MODEL, MAX_LLM_SPEND_USD)
    df = ba.review_ambiguous(df, reviewer, TARGET, SITE_TOPIC, BATCH_SIZE, MAX_LLM_ROWS)
    llm_log = pd.DataFrame(reviewer.cost_log)
    print(f"Claude spent this run: ${reviewer.spent:.4f}"
          + ("  (sample mode: cost_usd shows what each call cost when collected)" if USE_SAMPLE else ""))
    display(llm_log)
    display(ba.share_table(df.loc[df["llm_verdict"].notna(), "llm_verdict"], "llm_verdict"))
else:
    llm_log = pd.DataFrame()
    for c in ("llm_verdict", "llm_reason", "llm_confidence"):
        df[c] = None
    print("USE_CLAUDE = False: ambiguous rows stay in Neutral, marked 'not reviewed'.")

  batch 1: 25 rows, 25 reviews back  (spent $0.0000)
  batch 2: 25 rows, 25 reviews back  (spent $0.0000)
  batch 3: 14 rows, 14 reviews back  (spent $0.0000)
Claude spent this run: $0.0000  (sample mode: cost_usd shows what each call cost when collected)


,endpoint,label,model,input_tokens,output_tokens,cost_usd,source
0,messages,25 rows,claude-haiku-4-5-20251001,4104,984,0.009024,sample replay (cost when collected)
1,messages,25 rows,claude-haiku-4-5-20251001,4048,1078,0.009438,sample replay (cost when collected)
2,messages,14 rows,claude-haiku-4-5-20251001,2625,603,0.005640,sample replay (cost when collected)


,llm_verdict,count,share
0,keep_valuable,26,0.406
1,investigate,16,0.250
2,ignore_spam,11,0.172
3,keep_neutral,11,0.172


## 10. Four buckets

* **Valuable**: real, relevant sites. Thank them, build on the relationship, and use them to find look-alike prospects.
* **Neutral**: real but low value, plus your own sites. Nothing to do.
* **Likely spam**: automated spam. Understand it, then leave it alone. Disavow only if you have a manual action or you took part in a link scheme.
* **Investigate**: a human should look. On a site that sells courses or templates this is where piracy and reseller pages land.

`decided_by` tells you whether the rules or Claude placed each domain, and `why` gives the reason.

In [13]:
df = ba.final_buckets(df)
display(pd.crosstab(df["bucket"], df["decided_by"], margins=True))
show = ["domain_from", "page_from_title", "anchor", "dofollow", "domain_from_rank", "backlink_spam_score",
        "relevance", "group_count", "decided_by", "why"]
print("Valuable, strongest first:")
df[df["bucket"] == "Valuable"].sort_values("domain_from_rank", ascending=False)[show].head(15)

decided_by,claude,rules,rules (own site),All
bucket,,,,
Investigate,16,0,0,16
Likely spam,11,202,0,213
Neutral,11,0,3,14
Valuable,26,19,0,45
All,64,221,3,288


Valuable, strongest first:


,domain_from,page_from_title,anchor,dofollow,domain_from_rank,backlink_spam_score,relevance,group_count,decided_by,why
36,wix.com,Combine semantic & traditional keyword research for bett...,template in Google Sheets,True,696,5,0.452,4,rules,"rank 696, spam 5, relevance 0.45"
76,webtan.impress.co.jp,検索戦略、SEOのレポーティングと測定など： 2026年のトレンドを専門家20人が予測（後編） | Moz - ...,機械学習入門,True,589,0,0.217,3,claude,Japanese SEO/marketing publication; on-topic expert roun...
6,moz.com,2026 SEO Trends: Top Predictions from 20 Industry Expert...,Intro to Machine Learning,True,540,0,0.486,2,rules,"rank 540, spam 0, relevance 0.49"
13,searchenginejournal.com,Why Is Organic Traffic Down? Here's How To Segment The Data,ML script to group the keywords into topic clusters,True,523,0,0.342,2,rules,"rank 523, spam 0, relevance 0.34"
172,seohappyhour.com.dream.website,O que são e como usar palavras-chave semânticas em SEO,API de Linguagem Natural do Google,True,518,0,0.430,2,rules,"rank 518, spam 0, relevance 0.43"
48,seroundtable.com,"Daily Search Forum Recap: December 9, 2024",Different ways to Map Keywords to topics - with Supervis...,True,471,0,0.487,6,rules,"rank 471, spam 0, relevance 0.49"
63,mikekappel.com,Algorithmic Gatekeeping and Cognitive Liberty: The Archi...,https://academy.mlforseo.com/course/ai-search-llms-entit...,True,430,0,0.581,2,claude,On-topic article about AI and search ranking algorithms
25,autopod.co,Embeddingi dla Marketerów: Mapowanie Przestrzeni Tematyc...,www.mlforseo.com,True,386,15,0.341,58,claude,Polish marketing content on embeddings and semantic sear...
33,advancedwebranking.com,"Entity Search, Query Fan-Out, and Practical ML for SEO |...",one of the courses we have on the ML for SEO,True,385,0,0.569,4,claude,Advanced Web Ranking; on-topic article mentioning course...
72,hallam.agency,From keywords to credibility: A practical framework for ...,entity co-occurrence across the internet.,True,367,0,0.461,2,rules,"rank 367, spam 0, relevance 0.46"


In [14]:
print("Investigate:")
display(df[df["bucket"] == "Investigate"].sort_values("domain_from_rank", ascending=False)[show])
print("Likely spam that Claude caught (the rules left these as ambiguous):")
df[(df["bucket"] == "Likely spam") & (df["decided_by"] == "claude")][show].head(12)

Investigate:


,domain_from,page_from_title,anchor,dofollow,domain_from_rank,backlink_spam_score,relevance,group_count,decided_by,why
278,ja.majestic.com,@lazarinastoy - Author Summary - Majestic,https://academy.mlforseo.com/product/introduction-to-mac...,False,531,0,0.383,38,claude,Majestic author page; 38 links from domain; unusual volu...
9,mmocourse.hk,Download Lazarina Stoy – Introduction To Machine Learnin...,Lazarina Stoy – Introduction to Machine Learning Course,True,381,0,0.394,2,claude,MMO course site with 'download' in title; possible cours...
7,mmocourse.top,Download Lazarina Stoy – Semantic Keyword Research + AI ...,Lazarina Stoy – Semantic Keyword Research + AI Search & ...,True,359,15,0.541,2,claude,MMO course piracy site; 'download' indicator; spam score 15
14,mmocourse.org,Download Lazarina Stoy – Introduction To Machine Learnin...,Lazarina Stoy – Introduction to Machine Learning Course,True,349,0,0.383,2,claude,MMO course piracy site; 'download' course indicator
2,growthmentor.com,"Lazarina Stoy - Marketing & ML Consultant, Trainer, Spea...",https://www.mlforseo.com/,True,311,25,0.308,3,claude,Growth Mentor profile; spam score 25; founder profile link
5,imarketing.courses,Lazarina Stoy – Semantic Keyword Research + AI Search & ...,Link,True,286,5,0.599,4,claude,iMarketing.courses reseller site; possible paid course l...
116,concresca.com,Algorithmic Gatekeeping and Cognitive Liberty: The Archi...,https://academy.mlforseo.com/course/ai-search-llms-entit...,True,265,40,0.550,1,claude,Concresca.com; spam score 40; replicates other site content
27,personwise.ai,SEO Metrics Measurement and Interpretation — Free intera...,Your rank tracker is lying to you about AI search - MLfo...,False,181,20,0.590,3,claude,Course site linking to MLforSEO course content; possible...
275,a-moz.groupbuyseo.org,2026 SEO Trends: Top Predictions from 20 Industry Expert...,Intro to Machine Learning,True,180,5,0.500,1,claude,Domain has 'groupbuyseo' suggesting content piracy/resel...
79,aiseocourse.review,"AI SEO Course Cost 2026: Free, One-Time & Monthly Pricing",AI Search Optimization & Agentic SEO,False,158,0,0.674,1,claude,Course review site with suspicious .review TLD; appears ...


Likely spam that Claude caught (the rules left these as ambiguous):


,domain_from,page_from_title,anchor,dofollow,domain_from_rank,backlink_spam_score,relevance,group_count,decided_by,why
18,server2.radiookapi.net,Google Sheets Templates For Daytrading Charts Craft Hub ...,NaN,True,468,0,0.157,1,claude,Unrelated topic (daytrading charts); misplaced link
26,domainanalysis.space,Premium Link Building Experts Helping Businesses Build H...,High Authority mlforseo.com Backlinks for Long Term SEO ...,True,122,55,0.356,1,claude,Link building service selling backlinks; high spam score...
44,bye.fyi,👲 Domain Report 👲,mlforseo.com,False,306,50,0.260,2,claude,Domain report tool; spam score 50; generic automated link
78,buzzshrink.website,❤️ URL Shared ❤️,mlforseo.com,False,273,50,0.281,3,claude,URL sharing site; spam score 50; generic automated link
87,screenshots.wiki,👲 Domain Report 👲,mlforseo.com,False,252,50,0.215,1,claude,Screenshots wiki tool; spam score 50; automated link
97,urls-shortener.eu,❤️ URL Shared ❤️,mlforseo.com,False,245,50,0.175,2,claude,URL shortener; spam score 50; generic automated link
199,shortenurls.eu,❤️ URL Shared ❤️,mlforseo.com,False,263,50,0.227,2,claude,URL shortener tool; spam score 50; generic automated link
251,ready.pro,✅ Website Stats 📊,mlforseo.com,False,209,50,0.345,1,claude,Website stats aggregator; high spam score and generic di...
257,rankvanceauthority.info,"&##x1f680; Boost Organic Traffic, Rankings & Authority w...",mlforseo.com,True,84,50,0.207,1,claude,Backlink selling service with high spam score; clear SEO...
270,tunca.org,Where to buy 🚀 aged domains and backlinks 🔥 | 1025-4941,mlforseo.com,True,0,50,0.135,1,claude,Backlink marketplace with high spam score; automated spa...


In [15]:
spam = df[df["bucket"] == "Likely spam"]
print(f"{len(spam)} spam domains send {int(spam['group_count'].sum())} of the "
      f"{int(df['group_count'].sum())} links in this table.")
print("The spam networks, grouped by the page title they share:")
(spam.assign(title=spam["page_from_title"].fillna("").str[:60])
     .groupby("title").agg(domains=("domain_from", "count"), links=("group_count", "sum"),
                           median_spam_score=("backlink_spam_score", "median"),
                           median_relevance=("relevance", "median"))
     .sort_values("domains", ascending=False).head(10))

213 spam domains send 1456 of the 2596 links in this table.
The spam networks, grouped by the page title they share:


,domains,links,median_spam_score,median_relevance
title,,,,
"TELEGRAM @SEO_ANOMALY - SEO BACKLINKS, BLACK-LINKS, TRAFFIC",104,441,60.0,0.2535
🏆🏆Boost your Google rankings with Premium PBN & Link Buildin,24,31,60.0,0.3700
"TELEGRAM @SEO_ANOMALY | SEO BACKLINKS, PBN, TRAFFIC BOOST, L",20,38,95.0,0.3375
Where to buy 🚀 aged domains and backlinks 🔥 | 1025-4941,4,6,55.0,0.1535
Where to buy 🚀 aged domains and backlinks 🔥 from all-aged-do,4,4,65.0,0.1560
❤️ URL Shared ❤️,3,7,50.0,0.2270
👲 Domain Report 👲,3,6,50.0,0.2150
seonix.agency ★★★★★ 🚀 The Transformation Was Amazing! When I,2,2,70.0,0.4505
Best Tool for Pinterest SiteToSocial Fully Automated AI Gene,2,2,80.0,0.3600


## 11. Anchors

The anchors endpoint covers **every** live backlink, so a network with hundreds of identical anchors weighs a lot here. The anchor-type table counts backlinks, not domains.

In [16]:
anchors = ba.anchors_df(raw["anchors"])
anchors["anchor_type"] = [ba.anchor_type(a, None, [b.lower() for b in BRAND_TERMS],
                                         [k.lower() for k in KEYWORD_TERMS]) for a in anchors["anchor"]]
display(ba.share_table(anchors["anchor_type"], "anchor_type", weights=anchors["backlinks"]))
anchors.head(12)

,anchor_type,count,share
0,spam_phrase,1139,0.451
1,branded,633,0.251
2,image_or_empty,429,0.170
3,naked_url,160,0.063
4,keyword,112,0.044
5,other,38,0.015
6,generic,15,0.006


,anchor,backlinks,referring_domains,spam_score,rank,first_seen,anchor_type
0,NaN,429,10,0,156,2024-01-31,image_or_empty
1,"TELEGRAM @SEO_ANOMALY - SEO BACKLINKS, BLACK-LINKS, TRAF...",376,104,61,0,2025-10-28,spam_phrase
2,"@SEO_CARTEL IN TELEGRAM - SEO BACKLINKS, BULK LINK POSTI...",359,27,59,0,2026-05-20,spam_phrase
3,TG @BHS_LINKS - BEST SEO LINKS - https://telegram.me/bhs...,295,10,60,0,2026-02-23,spam_phrase
4,MLforSEO ✨,248,1,0,133,2024-05-21,branded
5,Subscribe to MLforSEO ✨,121,1,0,133,2024-05-21,branded
6,Join the MLforSEO academy,119,1,0,132,2025-02-21,branded
7,www.mlforseo.com,60,1,3,99,2026-06-18,naked_url
8,TG @BHS_LINKS - BEST SEO LINKS - https://t.me/bhs_links,48,6,60,0,2026-02-23,spam_phrase
9,MLforSEO,38,8,5,89,2025-04-15,branded


## 12. Link reclamation

`broken_backlinks` are live links pointing at URLs on your site that answered 4xx or 5xx when DataForSEO last crawled them. For each dead URL the notebook:

1. counts the broken links and linking domains,
2. re-checks the URL today (`CHECK_STATUS_NOW`, free), because DataForSEO's crawl can be weeks old and you may have fixed it already,
3. suggests a redirect target: the live sitemap URL whose path words are closest to the dead path (`rapidfuzz` `token_set_ratio`, 0-100; suggestions under 60 are left blank).

Always eyeball a suggestion before adding a 301. A high string score means similar words, not the same content.

In [17]:
broken = ba.backlinks_df(raw["broken_backlinks"])
if USE_SAMPLE:
    live_urls = pd.read_csv("sample_data/sitemap_urls.csv")["url"].tolist()
    status_now = pd.read_csv("sample_data/url_status_check.csv")
else:
    live_urls = ba.fetch_sitemap_urls(SITEMAP_URL)
    status_now = (ba.check_status(broken["url_to"].dropna().unique().tolist()) if CHECK_STATUS_NOW and len(broken)
                  else pd.DataFrame(columns=["url_to", "status_now", "final_url_now", "redirected_now"]))
print(f"{len(broken)} broken backlinks, {broken['url_to'].nunique() if len(broken) else 0} dead target URLs, "
      f"{len(live_urls)} URLs in the sitemap")
reclaim = ba.suggest_redirects(broken, live_urls)
if len(reclaim):
    reclaim = ba.reclamation_actions(reclaim.merge(status_now, on="url_to", how="left"))
reclaim

33 broken backlinks, 2 dead target URLs, 532 URLs in the sitemap


,url_to,broken_backlinks,referring_domains,status_code,best_domain_rank,example_linking_domain,dofollow_links,in_sitemap,suggested_redirect,match_score,status_now,final_url_now,redirected_now,action
0,https://www.mlforseo.com/resource/30-day-agent-search-op...,3,2,404,582,mlforseo-newsletter.kit.com,3,True,https://www.mlforseo.com/courses/ai-search-optimisation-...,61,200,https://www.mlforseo.com/resource/30-day-agent-search-op...,False,loads today (fixed since DataForSEO's crawl): nothing to do
1,https://www.mlforseo.com/resource/complete-guide-to-ml-c...,30,1,404,386,autopod.co,30,True,https://www.mlforseo.com/resource/complete-guide-underst...,88,200,https://www.mlforseo.com/resource/complete-guide-to-ml-c...,False,loads today (fixed since DataForSEO's crawl): nothing to do


## 13. Export

CSVs for each table, plus `backlink_audit.xlsx` with one tab per table. In Google Sheets use *File → Import → Upload*. In Colab, download from the folder icon on the left (`output/`). `cost_log.csv` holds every paid call from both APIs.

In [18]:
audit = df[[c for c in ba.AUDIT_COLS if c in df.columns]].sort_values(["bucket", "domain_from_rank"],
                                                                     ascending=[True, False])
bucket_counts = (df.groupby("bucket").agg(referring_domains=("domain_from", "count"),
                                          links=("group_count", "sum"))
                 .reset_index().sort_values("referring_domains", ascending=False))
full_cost_log = pd.concat([cost_log.assign(api="dataforseo"), llm_log.assign(api="anthropic")], ignore_index=True)
tables = {"audit": audit, "bucket_counts": bucket_counts, "summary": ba.summary_table(raw["summary"]),
          "referring_domains": rd, "anchors": anchors, "broken_backlinks": broken, "link_reclamation": reclaim}
for p in ba.export(tables, "output", xlsx_name="backlink_audit.xlsx"):
    print(p)
full_cost_log.to_csv("output/cost_log.csv", index=False)
if not USE_SAMPLE:  # keep what you need to re-run later without paying
    pd.Series(live_urls, name="url").to_csv("output/sitemap_urls.csv", index=False)
    status_now.to_csv("output/url_status_check.csv", index=False)
    if USE_CLAUDE:
        Path("output/claude_reviews.json").write_text(json.dumps(reviewer.raw, indent=1))
print("output/cost_log.csv")
print(f"\nTotal this run: DataForSEO ${cost_log['cost_usd'].sum() if not USE_SAMPLE else 0:.4f}, "
      f"Claude ${llm_log['cost_usd'].sum() if (USE_CLAUDE and not USE_SAMPLE) else 0:.4f}")
bucket_counts

output/audit.csv
output/bucket_counts.csv
output/summary.csv
output/referring_domains.csv
output/anchors.csv
output/broken_backlinks.csv
output/link_reclamation.csv
output/backlink_audit.xlsx
output/cost_log.csv

Total this run: DataForSEO $0.0000, Claude $0.0000


,bucket,referring_domains,links
1,Likely spam,213,1456
3,Valuable,45,191
0,Investigate,16,109
2,Neutral,14,840


## How to read the output

* **audit.csv**: one row per referring domain. `bucket`, `decided_by` and `why` first, then every feature. Filter `bucket = Investigate` first, then skim Valuable for outreach ideas.
* **bucket_counts.csv**: domains and links per bucket. `links` uses `group_count`, the number of links each domain sends.
* **link_reclamation.csv**: dead URLs with linking domains, today's status, a suggested redirect and the next step.
* **anchors.csv**, **referring_domains.csv**, **summary.csv**, **broken_backlinks.csv**: the flattened API data.
* **cost_log.csv**: what each call actually cost, from DataForSEO's `cost` field and Claude's token usage.

## Limits and honest caveats

* **Third-party index.** DataForSEO's index is its own crawl, not Google's. It will miss links Google knows about, list links Google ignores, and lag behind your site (the sample run found "broken" links to pages that load fine today).
* **One link per domain.** The audit judges each domain by its strongest link. A domain with one great link and fifty spammy ones is judged on the great one; check `group_count`.
* **Spam score and rank are DataForSEO's metrics**, not Google's. Spam networks can carry decent ranks (in the sample, one network's domains all sit at rank 149).
* **Relevance is a proxy** built from title, anchor and domain only, with an English-first model.
* **Claude only sees metadata.** Its verdicts are a second opinion on the ambiguous rows, not a site review. Spot-check them, especially anything you would act on. Verdicts can also change a little between runs on identical input (on the mlforseo.com data, two runs on identical input agreed on 54 of 64 rows; the 10 disagreements were all between neighbouring verdicts, such as neutral versus valuable, and none involved spam), so save the run you act on: `output/raw/claude/` keeps every response.
* **Disavow is rarely needed.** Google's guidance is to disavow only when you have a considerable number of bad links that have caused, or are likely to cause, a manual action.

## What to do next

* Pull the SERPs for the topics you want links around with [`dataforseo-serp-collection/`](https://github.com/lazarinastoy/mlforseo-code/tree/main/dataforseo-serp-collection), which uses the same DataForSEO client and guardrails.
* Turn your link reclamation list into redirects with the matching approach in [`fuzzy-matching-for-seo/`](https://github.com/lazarinastoy/mlforseo-code/tree/main/fuzzy-matching-for-seo).
* Go deeper on embeddings, classification and automation for SEO in the [MLforSEO Academy](https://academy.mlforseo.com/).